In [31]:
pip install -q langchain-groq langchain-community langchain-core requests duckduckgo-search


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [32]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
import requests

In [33]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

In [34]:
@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key=214c732878372744bdc9d6048c66025d&query={city}'

  response = requests.get(url)

  return response.json()

In [35]:
llm = ChatGroq(model="openai/gpt-oss-120b")


In [36]:
%pip install -U langchain-classic langchainhub


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [37]:
from langchain_classic.agents import create_react_agent, AgentExecutor

In [38]:
from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True
)

print(prompt)

input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'


In [39]:
agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather_data],
    prompt=prompt,
)

In [40]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True,
    handle_parsing_errors=True,
)

In [47]:
# Step 5: Invoke
response = agent_executor.invoke({"input": "Find the capital of India, then find it's current weather condition"})
print(response)



> Entering new AgentExecutor chain...
Question: Find the capital of India, then find it's current weather condition
Thought: I know the capital of India is New Delhi. To provide the current weather condition, I will use the get_weather_data tool for New Delhi.
Action: get_weather_data
Action Input: New Delhi{'request': {'type': 'City', 'query': 'New Delhi, India', 'language': 'en', 'unit': 'm'}, 'location': {'name': 'New Delhi', 'country': 'India', 'region': 'Delhi', 'lat': '28.600', 'lon': '77.200', 'timezone_id': 'Asia/Kolkata', 'localtime': '2026-10-12 00:32', 'localtime_epoch': 1791765120, 'utc_offset': '5.50'}, 'current': {'observation_time': '07:02 PM', 'temperature': 25, 'weather_code': 116, 'weather_icons': ['https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0004_black_low_cloud.png'], 'weather_descriptions': ['Partly Cloudy '], 'astro': {'sunrise': '06:20 AM', 'sunset': '05:55 PM', 'moonrise': '07:47 AM', 'moonset': '06:40 PM', 'moon_phase': 'Waxing Crescen

In [46]:
response['output']

'The capital of India is **New\u202fDelhi**. According to the latest weather data, New\u202fDelhi is currently experiencing **partly cloudy** conditions with a temperature of **25\u202f°C** (feels like 26\u202f°C), humidity\u202f67\u202f%, wind from the east at 10\u202fkm/h, and moderate air‑quality levels.'